# Native CLM v0 M3W-0 — Root Write-Drift Counterfactual Restoration

Checkpoint-only diagnostic. No Native CLM training and no new formal seeds. It evaluates the already-published M3L-2 online-address treatment checkpoints under a frozen 2×2 root/descendant operator-restoration design.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

BRANCH = 'codex/native-clm-v0-m3w0-write-drift-restoration'
REPO = Path('/kaggle/working/mini-cells')
M1_DIR = Path('/kaggle/working/native-clm-v0-m1')
M1 = M1_DIR / 'final-model.pt'
DATA = Path('/kaggle/working/native-clm-m3w0-data')
CHECKPOINTS = Path('/kaggle/working/native-clm-m3w0-checkpoints')
OUT = REPO / 'artifacts/experiments/native-clm-v0-m3w0-write-drift-restoration'

def run(cmd, check=True, env=None):
    print('+', ' '.join(map(str, cmd)), flush=True)
    return subprocess.run(list(map(str, cmd)), check=check, env=env)

if not (REPO / '.git').exists():
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', 'https://github.com/ArcheLabs/mini-cells.git', REPO])
else:
    os.chdir(REPO)
    run(['git', 'fetch', '--no-tags', 'origin', f'+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}'])
    run(['git', 'checkout', BRANCH])
    run(['git', 'merge', '--ff-only', f'origin/{BRANCH}'])
os.chdir(REPO)
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[lm]'])
print('HEAD:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
from kaggle_secrets import UserSecretsClient
import torch

secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['GITHUB_TOKEN'] = secrets.get_secret('GITHUB_TOKEN')
assert os.environ['HF_TOKEN'], 'Missing HF_TOKEN'
assert os.environ['GITHUB_TOKEN'], 'Missing GITHUB_TOKEN'
assert torch.cuda.is_available(), 'CUDA required for canonical M3W-0'
assert torch.cuda.device_count() >= 2, f'M3W-0 canonical runner requires two GPUs, found {torch.cuda.device_count()}'
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# Exact M1 operator source. Pin to the already-published M3L-2 HF revision; SHA verification remains authoritative.
run([
    sys.executable, 'scripts/research/fetch_native_clm_v0_m1_checkpoint.py',
    '--repo-id', 'archelabsxyz/native-clm-v0',
    '--filename', 'final-model.pt',
    '--expected-sha256', '91cc66f744c97e50105acbb7cdc328a95cb87a32c49baf5b0d6e462d4d4c4c7f',
    '--revision', '2b6ac153e926f899f038ff02c8c10041baaacb4a',
    '--output', M1,
])
print((M1_DIR / 'provenance.json').read_text())

In [ ]:
# Reconstruct the exact published M3L-2 snapshot; M3W-0 uses evaluation files only.
run([sys.executable, 'scripts/research/prepare_native_clm_v0_m3l2_data.py', '--output-dir', DATA])
manifest = json.loads((DATA / 'manifest.json').read_text())
print(json.dumps({'format': manifest['format'], 'dataset_revisions': manifest['dataset_revisions']}, indent=2))

In [ ]:
# Fetch only the three already-consumed M3L-2 online-address treatment checkpoints.
run([
    sys.executable, 'scripts/research/fetch_native_clm_v0_m3w0_checkpoints.py',
    '--output-dir', CHECKPOINTS,
])
print((CHECKPOINTS / 'manifest.json').read_text())

In [ ]:
# Run checkpoint-only 2×2 counterfactual restoration on two GPUs.
run([
    sys.executable, 'scripts/research/run_native_clm_v0_m3w0.py',
    '--m1-checkpoint', M1,
    '--checkpoints-dir', CHECKPOINTS,
    '--data-dir', DATA,
    '--output-dir', OUT,
    '--devices', '0,1',
])
result = json.loads((OUT / 'diagnostic-result.json').read_text())
print(json.dumps({
    'classification': result['classification'],
    'scientific_decision': result['scientific_decision'],
    'native_clm_training': result['native_clm_training'],
    'new_formal_seeds_consumed': result['new_formal_seeds_consumed'],
}, indent=2))
for seed in result['seed_results']:
    print('seed', seed['seed'], 'root=', seed['A_root_fraction'], 'desc=', seed['A_descendant_fraction'], 'gain kept=', seed['root_restore_new_domain_gain_retention'])

In [ ]:
# Publish lightweight evidence only; no checkpoints are generated or uploaded.
run([
    sys.executable, 'scripts/research/publish_native_clm_v0_m3w0.py',
    '--branch', BRANCH,
    '--output-dir', OUT,
])
print('Published M3W-0 classification:', result['classification'])